# DE closeout — four approved addendum requirements
Run on the school **CPU** after installation, with PostgreSQL running. This records the explicit approval already given by Ibrahim for the four separate signing/returning requirements. No further approval is inferred.
Stop on the first error. Read `../README_DE_CLOSEOUT.md` for scope and recovery details.


In [1]:
from pathlib import Path
import os, subprocess, json
PROJECT=Path('/home/jovyan/project/georgia-bid-readiness')
PY=PROJECT/'.venv-de/bin/python'
assert PY.is_file(), 'Existing .venv-de environment missing; complete the previous setup first'
os.chdir(PROJECT)
def run(*args, env=None):
    subprocess.run([str(PY), *args], cwd=PROJECT, env=env, check=True)
approval=json.loads((PROJECT/'configs/split_closeout_approval.json').read_text())
print('Reviewer:', approval['reviewer'])
print('Approval:', approval['human_approval'])
for item in approval['splits']:
    print(item['split_id'], '—', item['payload']['requirement'])
run('scripts/run_de_pipeline.py','check')


Reviewer: Ibrahim Shittu
Approval: Approve all four
hotel-26-008-addendum-01-sign — Sign Addendum 1.
hotel-26-008-addendum-01-return — Return Addendum 1 with the proposal.
hotel-26-008-addendum-02-sign — Sign Addendum 2.
hotel-26-008-addendum-02-return — Return Addendum 2 with the proposal.
Run report: /home/jovyan/project/georgia-bid-readiness/reports/de_runs/20261025T195800163646Z/run_report.json


## 1. Back up before applying migration 005

In [2]:
run('-c', "import sys;sys.path.insert(0,'scripts');from de_backup import database_backup;from pathlib import Path;print(database_backup(Path('../georgia-bid-readiness-backups/de_completion'),verify=False))")
run('scripts/apply_split_closeout_schema.py')
run('scripts/apply_split_closeout_schema.py')


{'dump': '../georgia-bid-readiness-backups/de_completion/gbr_20261025T195809543083Z.dump', 'source_tables': {'analytics_gbr.dim_date': {'rows': 9, 'sha256': '28e5546543149d7270015a0b3ca7148dd5da31f2940f499d2d5db016cc7f1406'}, 'analytics_gbr.dim_document': {'rows': 15, 'sha256': '3260b80f71f8e4355d90deaea4b2aa6f8865b0b4b7dd1edc0d65e141d8578992'}, 'analytics_gbr.dim_document_version': {'rows': 15, 'sha256': '84efd2e42866c8caf83321d58207dd6fdb736580bc37f6035e59f2d3707fa221'}, 'analytics_gbr.dim_model_run': {'rows': 2, 'sha256': 'f14b7f0bbd245387a6123744a4db0a23f7f3d4f39678dc66e3ab005d16ef8a01'}, 'analytics_gbr.dim_solicitation': {'rows': 2, 'sha256': '83649f0c8ae22fab7c1eb3a2f5ee8293c609141e3d0114091ca5c93ed29ab4fa'}, 'analytics_gbr.fct_amendment_relation': {'rows': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, 'analytics_gbr.fct_canonical_assertion': {'rows': 14, 'sha256': '06f56609973e16476487f5c8f958f9ad23a3ca42c92a056eecbb277ec5a57959'}, 'analytics_

## 2. Run the regression checks
Expected: **48 passed**. Synthetic test records roll back.

In [3]:
env=os.environ.copy()
env.update(GBR_RUN_DB_TESTS='1', PYTEST_DISABLE_PLUGIN_AUTOLOAD='1')
run('-m','pytest','tests/test_requirement_loader.py','tests/test_canonical_requirements.py',
    'tests/test_review_importer.py','tests/test_de_completion.py','tests/test_split_closeout.py','-v',env=env)


============================= test session starts ==============================
platform linux -- Python 3.11.5, pytest-9.1.1, pluggy-1.7.0 -- /home/jovyan/project/georgia-bid-readiness/.venv-de/bin/python
cachedir: .pytest_cache
rootdir: /home/jovyan/project/georgia-bid-readiness
collecting ... collected 48 items

tests/test_requirement_loader.py::test_preflight_preserves_invalid_and_empty_outputs PASSED [  2%]
tests/test_requirement_loader.py::test_tampered_prompt_rejected PASSED   [  4%]
tests/test_requirement_loader.py::test_tampered_input_rejected PASSED    [  6%]
tests/test_requirement_loader.py::test_saved_flags_not_trusted PASSED    [  8%]
tests/test_requirement_loader.py::test_saved_messages_not_trusted PASSED [ 10%]
tests/test_requirement_loader.py::test_missing_job_rejected PASSED       [ 12%]
tests/test_requirement_loader.py::test_preview_rolls_back PASSED         [ 14%]
tests/test_requirement_loader.py::test_repeat_load_and_bad_evidence PASSED [ 16%]
tests/test_requiremen

## 3. Preview, commit the authorized approvals, and verify a repeat
The first preview/commit inserts four split reviews, four canonical groups and four split assertions. If already installed, the same records are reused. The original combined model candidates are retained as historical records.

In [4]:
run('scripts/closeout_split_requirements.py')
run('scripts/closeout_split_requirements.py','--commit')
result=subprocess.run([str(PY),'scripts/closeout_split_requirements.py','--commit'],cwd=PROJECT,check=True,capture_output=True,text=True)
repeat=json.loads(result.stdout)
assert not any(v for k,v in repeat['counts'].items() if k.endswith('_inserted')), repeat
print('Repeat safe:',repeat)


{
  "mode": "preview_rolled_back",
  "counts": {
    "split_requirement_reviews_inserted": 4,
    "canonical_requirements_inserted": 4,
    "canonical_split_assertions_inserted": 4
  }
}
{
  "mode": "committed",
  "counts": {
    "split_requirement_reviews_inserted": 4,
    "canonical_requirements_inserted": 4,
    "canonical_split_assertions_inserted": 4
  }
}
Repeat safe: {'mode': 'committed', 'counts': {'split_requirement_reviews_reused': 4, 'canonical_requirements_reused': 4, 'canonical_split_assertions_reused': 4}}


## 4. Refresh dbt, reports, and verified backups
Expected: 17 dbt models, 42 dbt tests. This cell can take several minutes.

In [5]:
run('scripts/run_de_pipeline.py','run')
run('scripts/export_de_closeout_report.py')
run('-c', "import sys;sys.path.insert(0,'scripts');from de_backup import archive_project;from pathlib import Path;print(archive_project(Path('../georgia-bid-readiness-backups/de_completion')))")


DE refresh, dbt tests, and restore verification passed.
Run report: /home/jovyan/project/georgia-bid-readiness/reports/de_runs/20261025T195818548645Z/run_report.json
{
  "status": "pilot_de_closeout_verified",
  "recorded_at": "20261025T195948969667Z",
  "metrics": {
    "solicitation_id": "commerce-ga-26-008",
    "title": "Hotel Feasibility Analysis",
    "documents": 3,
    "extraction_records": 19,
    "extraction_failures": 0,
    "unresolved_page_flags": 0,
    "distinct_requirements": 17,
    "current_source_assertions": 18,
    "historical_followup_candidates": 3,
    "followups_resolved_by_split": 3,
    "outstanding_followups": 0,
    "accepted_splits": 4,
    "pending_splits": 0
  },
  "scope": "Current hotel pilot reviewed requirement population only; not extraction completeness or model accuracy.",
  "history": "Three combined parent candidates retain needs_followup in immutable review history; all three are operationally resolved by four approved splits."
}
Report folder:

## 5. Inspect and download the closeout report
Expected hotel totals: 17 distinct requirements, 18 current assertions, four approved splits, zero pending splits, three parents resolved through splitting and zero outstanding follow-ups in the current population. These are not model accuracy or completeness metrics.

In [6]:
from IPython.display import display, HTML
from urllib.parse import quote
reports=sorted((PROJECT/'reports/de_closeout').glob('*/closeout_report.json'))
assert reports, 'No closeout report found'
latest=reports[-1]
report=json.loads(latest.read_text())
assert report['status']=='pilot_de_closeout_verified'
display(HTML(latest.with_suffix('.html').read_text()))
prefix=os.environ.get('JUPYTERHUB_SERVICE_PREFIX','/user/ishittu1@students.kennesaw.edu/').rstrip('/')
for path in [latest, latest.with_suffix('.html')]:
    assert path.is_file()
    url=prefix+'/files/'+quote(path.relative_to('/home/jovyan').as_posix(),safe='/')
    display(HTML(f'<a href="{url}" target="_blank">{path.name}</a>'))
pipeline_reports=sorted((PROJECT/'reports/de_runs').glob('*/run_report.json'))
full=[p for p in pipeline_reports if json.loads(p.read_text())['mode']=='run'][-1]
assert json.loads(full.read_text())['status']=='complete'
print('Pipeline status: complete')
print('Closeout status:',report['status'])
print('Backups:',PROJECT.parent/'georgia-bid-readiness-backups/de_completion')


solicitation_id,commerce-ga-26-008
title,Hotel Feasibility Analysis
documents,3
extraction_records,19
extraction_failures,0
unresolved_page_flags,0
distinct_requirements,17
current_source_assertions,18
historical_followup_candidates,3
followups_resolved_by_split,3
outstanding_followups,0


Pipeline status: complete
Closeout status: pilot_de_closeout_verified
Backups: /home/jovyan/project/georgia-bid-readiness-backups/de_completion


## 6. Save the completed closeout
Download the latest database dump and matching verification JSON, project ZIP, closeout report and pipeline report. Then run in the terminal:

```bash
git add .github/workflows/de-completion.yml README_DE_CLOSEOUT.md
git add configs/split_closeout_approval.json configs/split_closeout_approval.sha256
git add warehouse/ddl/005_split_closeout.sql
git add scripts/apply_split_closeout_schema.py scripts/closeout_split_requirements.py scripts/export_de_closeout_report.py
git add tests/test_split_closeout.py notebooks/11_de_closeout.ipynb
git add dbt_gbr/models/closeout_sources.yml dbt_gbr/models/fct_current_requirement_assertion.sql
git add dbt_gbr/models/fct_split_parent_resolution.sql dbt_gbr/models/mart_de_closeout.sql
git add dbt_gbr/models/mart_solicitation_overview.sql dbt_gbr/tests/closeout_reconciliation.sql
git add reports/de_closeout_validation
git diff --cached --stat
git commit -m "Close approved addendum splits with canonical lineage and dbt checks"
git push
```

Confirm GitHub Actions passes. You can then return to AI extraction improvement. Do not rerun the old notebook 10 count check expecting zero split approvals or 13 groups: those were the previous baseline.
